<a href="https://colab.research.google.com/github/Gagandeep1227/ML-2/blob/main/foil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
facts = {
    "parent": {
        ("alice", "bob"),
        ("alice", "carol"),
        ("bob", "david"),
        ("carol", "emma")
    },

    "female": {
        ("alice",),
        ("carol",),
        ("emma",)
    },

    "male": {
        ("bob",),
        ("david",)
    }
}

positive_examples = {
    ("alice", "david"),
    ("alice", "emma")
}

negative_examples = {
    ("alice", "bob"),
    ("bob", "emma"),
    ("carol", "david")
}


def parent(x, y):
    return (x, y) in facts["parent"]


def female(x):
    return (x,) in facts["female"]


def male(x):
    return (x,) in facts["male"]


candidate_literals = [
    "parent(X, Z)",
    "parent(Z, Y)",
    "female(X)",
    "female(Y)",
    "male(X)",
    "male(Y)"
]


def satisfies_literal(example, literal):
    """
    Check whether a literal is true for an example.

    For the grandparent example:
        X = first element
        Y = second element

    Z is searched for automatically.
    """

    X, Y = example

    if literal == "female(X)":
        return female(X)

    if literal == "female(Y)":
        return female(Y)

    if literal == "male(X)":
        return male(X)

    if literal == "male(Y)":
        return male(Y)

    # parent(X, Z) AND parent(Z, Y)
    if literal == "parent(X, Z)":
        return any(parent(X, z) for z in get_people())

    if literal == "parent(Z, Y)":
        return any(parent(z, Y) for z in get_people())

    return False


def get_people():
    people = set()

    for x, y in facts["parent"]:
        people.add(x)
        people.add(y)

    return people

class Rule:
    def __init__(self):
        self.conditions = []

    def add_condition(self, condition):
        self.conditions.append(condition)

    def covers(self, example):
        """
        Check whether the rule covers an example.
        """

        X, Y = example

        has_parent_xz = "parent(X, Z)" in self.conditions
        has_parent_zy = "parent(Z, Y)" in self.conditions

        if has_parent_xz and has_parent_zy:

            # There must exist the SAME Z satisfying both.
            for z in get_people():
                if parent(X, z) and parent(z, Y):
                    return True

            return False

        # If only parent(X,Z) exists
        if has_parent_xz:
            if not any(parent(X, z) for z in get_people()):
                return False

        # If only parent(Z,Y) exists
        if has_parent_zy:
            if not any(parent(z, Y) for z in get_people()):
                return False

        # Other conditions
        if "female(X)" in self.conditions:
            if not female(X):
                return False

        if "female(Y)" in self.conditions:
            if not female(Y):
                return False

        if "male(X)" in self.conditions:
            if not male(X):
                return False

        if "male(Y)" in self.conditions:
            if not male(Y):
                return False

        return True

    def __str__(self):
        if not self.conditions:
            return "grandparent(X, Y) :- TRUE"

        return "grandparent(X, Y) :- " + ", ".join(self.conditions)

def foil_gain(rule, literal, pos, neg):
    """
    Simplified FOIL Gain.

    We calculate how many positive and negative examples
    remain after adding a literal.

    gain = positive_remaining / total_remaining
    """

    old_covered_pos = [
        e for e in pos
        if rule.covers(e)
    ]

    old_covered_neg = [
        e for e in neg
        if rule.covers(e)
    ]

    # Temporarily add literal
    rule.add_condition(literal)

    new_covered_pos = [
        e for e in pos
        if rule.covers(e)
    ]

    new_covered_neg = [
        e for e in neg
        if rule.covers(e)
    ]

    # Remove temporary literal
    rule.conditions.pop()

    old_total = len(old_covered_pos) + len(old_covered_neg)
    new_total = len(new_covered_pos) + len(new_covered_neg)

    if new_total == 0:
        return 0

    old_probability = (
        len(old_covered_pos) / old_total
        if old_total else 0
    )

    new_probability = (
        len(new_covered_pos) / new_total
        if new_total else 0
    )

    return new_probability - old_probability


def foil(pos, neg):

    learned_rules = []

    pos = set(pos)
    neg = set(neg)

    while pos:

        rule = Rule()
        rule_neg = set(neg)

        print("\nStarting new rule")

        while rule_neg:

            best_literal = None
            best_gain = -1

            for literal in candidate_literals:

                if literal in rule.conditions:
                    continue

                gain = foil_gain(
                    rule,
                    literal,
                    pos,
                    rule_neg
                )

                print(
                    f"  Candidate: {literal:15} "
                    f"Gain = {gain:.3f}"
                )

                if gain > best_gain:
                    best_gain = gain
                    best_literal = literal

            if best_literal is None:
                break

            rule.add_condition(best_literal)

            # Keep only negative examples still covered
            rule_neg = {
                example
                for example in rule_neg
                if rule.covers(example)
            }

            print(
                f"  Added: {best_literal}"
            )

        learned_rules.append(rule)

        # Remove positive examples covered by this rule
        covered_pos = {
            example
            for example in pos
            if rule.covers(example)
        }

        pos -= covered_pos

        print("Learned:", rule)
        print("Covered positive examples:", covered_pos)

    return learned_rules


rules = foil(
    positive_examples,
    negative_examples
)

print("\n==============================")
print("FINAL LEARNED RULES")
print("==============================")

for rule in rules:
    print(rule)



Starting new rule
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(X)       Gain = 0.100
  Candidate: female(Y)       Gain = 0.100
  Candidate: male(X)         Gain = -0.400
  Candidate: male(Y)         Gain = -0.067
  Added: female(X)
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(Y)       Gain = 0.500
  Candidate: male(X)         Gain = 0.000
  Candidate: male(Y)         Gain = -0.167
  Added: female(Y)
Learned: grandparent(X, Y) :- female(X), female(Y)
Covered positive examples: {('alice', 'emma')}

Starting new rule
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Candidate: female(X)       Gain = 0.083
  Candidate: female(Y)       Gain = -0.250
  Candidate: male(X)         Gain = -0.250
  Candidate: male(Y)         Gain = 0.083
  Added: female(X)
  Candidate: parent(X, Z)    Gain = 0.000
  Candidate: parent(Z, Y)    Gain = 0.000
  Ca

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

Saving Assign.csv to Assign (4).csv
Dataset loaded successfully!
Shape: (6, 1)


,| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |
0,| ---------- | ----------- | ---------- | ----...
1,| High | High | Good | High...
2,| Low | Low | Poor | Low ...
3,| High | Medium | Good | High...
4,| Medium | Low | Average | Medi...


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nClass distribution:")
print(df.iloc[:, -1].value_counts())

Columns:
['| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 1 columns):
 #   Column                                                                               Non-Null Count  Dtype 
---  ------                                                                               --------------  ----- 
 0   | Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |  6 non-null      object
dtypes: object(1)
memory usage: 180.0+ bytes
None

Missing values:
| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |    0
dtype: int64

Class distribution:
| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |
| ---------- | ----------- | ---------- | -------------- | ------------- | ------ |    1
| High       | High        | Good       | High           | Good          | Pass

In [ ]:
# Make a copy
data = df.copy()

# Remove rows containing missing values
data = data.dropna().reset_index(drop=True)

# Convert all columns to strings
for col in data.columns:
    data[col] = data[col].astype(str)

print("Preprocessed dataset:")
display(data.head())

Preprocessed dataset:


,| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |
0,| ---------- | ----------- | ---------- | ----...
1,| High | High | Good | High...
2,| Low | Low | Poor | Low ...
3,| High | Medium | Good | High...
4,| Medium | Low | Average | Medi...


In [ ]:
!pip -q install python-weka-wrapper3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 77.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 27.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

target = data.columns[-1]
features = list(data.columns[:-1])

print("Target column:", target)
print("Features:", features)

Target column: | Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |
Features: []


In [ ]:
def rule_accuracy(df, conditions, target_col, target_value):
    """
    Calculate how well a rule predicts target_value.
    """
    subset = df.copy()

    for feature, value in conditions.items():
        subset = subset[subset[feature] == value]

    if len(subset) == 0:
        return 0, 0, 0

    correct = (subset[target_col] == target_value).sum()

    return correct / len(subset), correct, len(subset)


def generate_ripper_rules(df, target_col):
    rules = []

    classes = df[target_col].unique()

    for target_value in classes:

        remaining = df[df[target_col] == target_value].copy()

        while len(remaining) > 0:

            best_condition = None
            best_score = -1

            # Search for the best single condition
            for feature in df.columns:
                if feature == target_col:
                    continue

                for value in df[feature].unique():

                    subset = df[df[feature] == value]

                    if len(subset) == 0:
                        continue

                    correct = (subset[target_col] == target_value).sum()
                    precision = correct / len(subset)

                    if precision > best_score:
                        best_score = precision
                        best_condition = (feature, value)

            if best_condition is None:
                break

            feature, value = best_condition

            # Create rule
            conditions = {feature: value}

            subset = df[df[feature] == value]

            correct = (subset[target_col] == target_value).sum()

            if correct == 0:
                break

            rules.append({
                "conditions": conditions,
                "prediction": target_value,
                "accuracy": best_score,
                "covered": len(subset)
            })

            # Remove covered positive examples
            covered_positive = (
                (remaining[feature] == value)
            )

            if covered_positive.sum() == 0:
                break

            remaining = remaining[~covered_positive]

            if len(remaining) == 0:
                break

    return rules

In [ ]:
ripper_rules = generate_ripper_rules(data, target)

print("Generated RIPPER-style Rules:\n")

for i, rule in enumerate(ripper_rules, 1):

    conditions = " AND ".join(
        [f"{k} = {v}" for k, v in rule["conditions"].items()]
    )

    print(
        f"Rule {i}: IF {conditions} "
        f"THEN {target} = {rule['prediction']} "
        f"(accuracy={rule['accuracy']:.2f})"
    )

Generated RIPPER-style Rules:



In [ ]:
# Create symbolic examples
examples = []

for index, row in data.iterrows():

    example = {
        "id": index,
        "attributes": {
            col: row[col]
            for col in features
        },
        "target": row[target]
    }

    examples.append(example)

print("Number of examples:", len(examples))

print("\nFirst example:")
print(examples[0])

Number of examples: 6

First example:
{'id': 0, 'attributes': {}, 'target': '| ---------- | ----------- | ---------- | -------------- | ------------- | ------ |'}


In [ ]:
import math

def foil_gain(p0, n0, p1, n1, t):

    if p0 + n0 == 0:
        return 0

    if p1 + n1 == 0:
        return 0

    if p0 == 0 or p1 == 0:
        return 0

    old_ratio = p0 / (p0 + n0)
    new_ratio = p1 / (p1 + n1)

    if old_ratio == 0 or new_ratio == 0:
        return 0

    return t * (
        math.log2(new_ratio) -
        math.log2(old_ratio)
    )

In [ ]:
def candidate_literals(df, features):

    candidates = []

    for feature in features:

        for value in df[feature].unique():

            candidates.append(
                (feature, value)
            )

    return candidates

In [ ]:
def apply_conditions(df, conditions):

    result = df.copy()

    for feature, value in conditions:
        result = result[
            result[feature] == value
        ]

    return result


def find_best_literal(
    df,
    conditions,
    target_col,
    target_value,
    features
):

    current = apply_conditions(
        df,
        conditions
    )

    p0 = (
        current[target_col] == target_value
    ).sum()

    n0 = (
        current[target_col] != target_value
    ).sum()

    best_literal = None
    best_gain = -float("inf")

    candidates = candidate_literals(
        df,
        features
    )

    for literal in candidates:

        # Don't add the same condition twice
        if literal in conditions:
            continue

        new_conditions = conditions + [literal]

        new_subset = apply_conditions(
            df,
            new_conditions
        )

        p1 = (
            new_subset[target_col] == target_value
        ).sum()

        n1 = (
            new_subset[target_col] != target_value
        ).sum()

        if p1 == 0:
            continue

        t = p1

        gain = foil_gain(
            p0,
            n0,
            p1,
            n1,
            t
        )

        if gain > best_gain:

            best_gain = gain
            best_literal = literal

    return best_literal, best_gain

In [ ]:
def foil(df, target_col):

    learned_rules = []

    classes = df[target_col].unique()

    features = [
        col for col in df.columns
        if col != target_col
    ]

    for target_value in classes:

        positive = df[
            df[target_col] == target_value
        ].copy()

        negative = df[
            df[target_col] != target_value
        ].copy()

        print("\nLearning rules for:", target_value)

        while len(positive) > 0:

            conditions = []

            # Start with all examples
            current = df.copy()

            while True:

                covered = apply_conditions(
                    df,
                    conditions
                )

                neg_covered = covered[
                    covered[target_col] != target_value
                ]

                # Rule contains no negative examples
                if len(neg_covered) == 0:

                    if len(covered) > 0:
                        break

                # No more literals available
                literal, gain = find_best_literal(
                    df,
                    conditions,
                    target_col,
                    target_value,
                    features
                )

                if literal is None:
                    break

                # Add best literal
                conditions.append(literal)

                print(
                    "  Added literal:",
                    literal,
                    "FOIL Gain:",
                    round(gain, 4)
                )

            # Check whether rule covers positive examples
            covered = apply_conditions(
                df,
                conditions
            )

            covered_positive = covered[
                covered[target_col] == target_value
            ]

            if len(covered_positive) == 0:
                break

            # Save rule
            learned_rules.append({
                "target": target_value,
                "conditions": conditions,
                "covered": len(covered_positive)
            })

            # Remove covered positive examples
            positive = positive[
                ~positive.index.isin(
                    covered_positive.index
                )
            ]

    return learned_rules

In [ ]:
foil_rules = foil(data, target)


Learning rules for: | ---------- | ----------- | ---------- | -------------- | ------------- | ------ |

Learning rules for: | High       | High        | Good       | High           | Good          | Pass   |

Learning rules for: | Low        | Low         | Poor       | Low            | Poor          | Fail   |

Learning rules for: | High       | Medium      | Good       | High           | Good          | Pass   |

Learning rules for: | Medium     | Low         | Average    | Medium         | Average       | Fail   |

Learning rules for: | High       | High        | Good       | High           | Average       | Pass   |


In [ ]:
print("\n" + "=" * 60)
print("FINAL FOIL RULES")
print("=" * 60)

for i, rule in enumerate(foil_rules, 1):

    conditions = " AND ".join(
        [
            f"{feature}(X, {value})"
            for feature, value in rule["conditions"]
        ]
    )

    print(
        f"\nRule {i}:"
    )

    print(
        f"{target}(X, {rule['target']}) :- "
        f"{conditions}."
    )


FINAL FOIL RULES

Rule 1:
| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | ---------- | ----------- | ---------- | -------------- | ------------- | ------ |) :- .

Rule 2:
| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | High       | High        | Good       | High           | Good          | Pass   |) :- .

Rule 3:
| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | Low        | Low         | Poor       | Low            | Poor          | Fail   |) :- .

Rule 4:
| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | High       | Medium      | Good       | High           | Good          | Pass   |) :- .

Rule 5:
| Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | Medium     | Low         | Average    | Medium         | Average       | Fail   |) :- .

Rule 6:
| Attendance | Study_Hours | Assignment | I

In [ ]:
print("=" * 70)
print("RIPPER-STYLE CLASSIFICATION RULES")
print("=" * 70)

for i, rule in enumerate(ripper_rules, 1):

    conditions = " AND ".join(
        [
            f"{k} = {v}"
            for k, v in rule["conditions"].items()
        ]
    )

    print(
        f"Rule {i}: IF {conditions} "
        f"THEN {target} = {rule['prediction']}"
    )


print("\n")
print("=" * 70)
print("FOIL FIRST-ORDER RULES")
print("=" * 70)

for i, rule in enumerate(foil_rules, 1):

    conditions = " AND ".join(
        [
            f"{feature}(X, {value})"
            for feature, value in rule["conditions"]
        ]
    )

    print(
        f"Rule {i}: "
        f"{target}(X, {rule['target']}) :- "
        f"{conditions}."
    )

RIPPER-STYLE CLASSIFICATION RULES


FOIL FIRST-ORDER RULES
Rule 1: | Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | ---------- | ----------- | ---------- | -------------- | ------------- | ------ |) :- .
Rule 2: | Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | High       | High        | Good       | High           | Good          | Pass   |) :- .
Rule 3: | Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | Low        | Low         | Poor       | Low            | Poor          | Fail   |) :- .
Rule 4: | Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | High       | Medium      | Good       | High           | Good          | Pass   |) :- .
Rule 5: | Attendance | Study_Hours | Assignment | Internal_Marks | Participation | Result |(X, | Medium     | Low         | Average    | Medium         | Average       | Fail   |) :- .
Rule 6: | Attend